In [37]:
# ============================================================
# BONUS CHALLENGE: Deploying an ADK Multi-Agent System to Vertex AI Agent Engine
# Step 1: Initialize Vertex AI & Regional Cloud Storage
# Step 2: Define Agent Hierarchy using Google Agent Development Kit (ADK)
# ============================================================
import os
import warnings

from google.adk.agents.llm_agent import LlmAgent
from google.adk.agents.sequential_agent import SequentialAgent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool
from google.cloud import storage
import requests
import vertexai

# Disable LiteLLM MCP subsystem
os.environ["LITELLM_DISABLE_MCP"] = "true"

# Suppress experimental JSON schema warnings
warnings.filterwarnings(
    "ignore",
    message=".*JSON_SCHEMA_FOR_FUNC_DECL is enabled.*",
)

# ------------------------------------------------------------
# STEP 1: INITIALIZE VERTEX AI FOR AGENT ENGINE
# ------------------------------------------------------------
PROJECT_ID_LIST = !gcloud config get-value project
PROJECT_ID = PROJECT_ID_LIST[0] if PROJECT_ID_LIST else ""
if PROJECT_ID == "(unset)" or not PROJECT_ID:
    PROJECT_ID = "qwiklabs-gcp-01-6d489226baf6"

# Agent Engine requires a regional endpoint (us-central1)
LOCATION = "us-central1"
BUCKET_NAME = f"agent-engine-staging-{PROJECT_ID}"
STAGING_BUCKET = f"gs://{BUCKET_NAME}"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_QUOTA_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["ADK_SUPPRESS_GEMINI_LITELLM_WARNINGS"] = "true"

# Ensure the Cloud Storage staging bucket exists before deployment
storage_client = storage.Client(project=PROJECT_ID)
bucket = storage_client.bucket(BUCKET_NAME)
if not bucket.exists():
    print(f"Creating staging bucket: {STAGING_BUCKET} in {LOCATION}...")
    storage_client.create_bucket(bucket, location=LOCATION)
else:
    print(f"Using existing staging bucket: {STAGING_BUCKET}")

# Initialize Vertex AI with project, location, and staging bucket
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)

print(f"Vertex AI initialized successfully:")
print(f"  Project ID     : {PROJECT_ID}")
print(f"  Location       : {LOCATION}")
print(f"  Staging Bucket : {STAGING_BUCKET}")

# ------------------------------------------------------------
# STEP 2: DEFINE CUSTOM TOOLS & MULTI-AGENT WORKFLOW
#   Hierarchy:
#   root_agent -> [greeter_agent, weather_agent,
#                  answer_team (search_agent -> critique_agent -> refine_agent)]
# ------------------------------------------------------------

def geocode_place_free(address: str) -> str:
    """Geocode an address into latitude and longitude coordinates."""
    base_url = "https://nominatim.openstreetmap.org/search"
    params = {"q": address, "format": "json", "limit": 1}
    headers = {
        "User-Agent": "RussellWeatherApp/1.0 (russell.f.zimmermann@saic.com)"
    }
    try:
        response = requests.get(
            base_url, params=params, headers=headers, timeout=10
        )
        response.raise_for_status()
        data = response.json()
        if data:
            return f"{data[0]['lat']}, {data[0]['lon']}"
    except Exception:
        pass
    return "Error: Location not found."

def get_nws_forecast(latitude: float, longitude: float) -> str:
    """Retrieve the National Weather Service (NWS) forecast."""
    headers = {
        "User-Agent": "RussellWeatherApp/1.0 (russell.f.zimmermann@saic.com)"
    }
    points_url = f"https://api.weather.gov/points/{latitude},{longitude}"
    try:
        points_response = requests.get(
            points_url, headers=headers, timeout=10
        )
        points_response.raise_for_status()
        forecast_url = points_response.json()["properties"]["forecast"]
        forecast_response = requests.get(
            forecast_url, headers=headers, timeout=10
        )
        forecast_response.raise_for_status()
        periods = forecast_response.json()["properties"]["periods"]
        formatted = []
        for period in periods[:2]:
            formatted.append(
                f"{period['name']}: {period['temperature']}°"
                f"{period['temperatureUnit']}, {period['shortForecast']}."
            )
        return "\n\n".join(formatted)
    except Exception as e:
        return f"Error: {e}"

MODEL_NAME = "gemini-2.5-flash"

greeter_agent = LlmAgent(
    name="greeter_agent",
    model=MODEL_NAME,
    description="Greets the user and explains what the system can do.",
    instruction=(
        "You are a friendly Greeter Agent. Warmly welcome the user, "
        "introduce yourself, and explain that this system can provide US "
        "weather forecasts or answer complex questions using a 3-step "
        "Search, Critique, and Refine workflow team."
    ),
)

weather_agent = LlmAgent(
    name="weather_agent",
    model=MODEL_NAME,
    description="Retrieves US weather forecasts given a city or location.",
    instruction=(
        "You are a helpful weather assistant. When a user provides a "
        "location, use your geocoding and forecast tools to find the "
        "weather forecast for that location. Always explicitly state "
        "the identified state and country in your final response."
    ),
    tools=[geocode_place_free, get_nws_forecast],
)

# Isolated Google Search worker wrapped in AgentTool to prevent tool collisions
google_search_worker = LlmAgent(
    name="google_search_worker",
    model=MODEL_NAME,
    description="Executes Google Search queries to retrieve factual data.",
    instruction=(
        "Use the google_search tool to find accurate, up-to-date facts "
        "and data for the given query."
    ),
    tools=[google_search],
)

search_agent = LlmAgent(
    name="search_agent",
    model=MODEL_NAME,
    description="Finds data using Google Search to answer the question.",
    instruction=(
        "You are the Search Agent (Step 1 of the Answer Team). "
        "Use your `google_search_worker` tool to find accurate, up-to-date "
        "data to answer the user's question, and provide a comprehensive "
        "initial draft answer based on the search findings."
    ),
    tools=[AgentTool(agent=google_search_worker)],
    output_key="initial_answer",
)

critique_agent = LlmAgent(
    name="critique_agent",
    model=MODEL_NAME,
    description="Reviews the initial response and suggests improvements.",
    instruction=(
        "You are the Critique Agent (Step 2 of the Answer Team). "
        "Review the initial response generated by the Search Agent: "
        "{initial_answer}\n\n"
        "Evaluate its accuracy, completeness, structure, and clarity. "
        "Provide 2-3 specific, constructive suggestions on how the response "
        "can be improved before returning it to the user."
    ),
    output_key="critique_feedback",
)

refine_agent = LlmAgent(
    name="refine_agent",
    model=MODEL_NAME,
    description="Rewrites the response based on Critique Agent's suggestions.",
    instruction=(
        "You are the Refine Agent (Step 3 of the Answer Team). "
        "Take the Search Agent's initial response:\n{initial_answer}\n\n"
        "And the Critique Agent's suggestions:\n{critique_feedback}\n\n"
        "Rewrite and polish the response so it directly, clearly, and "
        "accurately answers the user's question incorporating the suggested "
        "improvements. Return the final refined response for the user."
    ),
    output_key="final_refined_answer",
)

answer_team = SequentialAgent(
    name="answer_team",
    description=(
        "Sequential workflow team that searches, critiques, and refines "
        "answers to questions."
    ),
    sub_agents=[search_agent, critique_agent, refine_agent],
)

root_agent = LlmAgent(
    name="root_agent",
    model=MODEL_NAME,
    description=(
        "Root coordinating agent that delegates to greeter_agent, "
        "weather_agent, and answer_team."
    ),
    instruction=(
        "You are the Root Coordinating Agent. Route user requests to the "
        "appropriate sub-agent:\n"
        "- If the user is saying hello, greeting you, or asking what you "
        "can do, delegate to `greeter_agent`.\n"
        "- If the user asks about US weather, temperature, or forecasts, "
        "delegate to `weather_agent`.\n"
        "- For all general knowledge, factual, research, or current events "
        "questions, delegate to the `answer_team` sequential workflow agent."
    ),
    sub_agents=[greeter_agent, weather_agent, answer_team],
)

print("\nMulti-Agent Hierarchy defined successfully: root_agent is ready.")

Using existing staging bucket: gs://agent-engine-staging-qwiklabs-gcp-01-6d489226baf6
Vertex AI initialized successfully:
  Project ID     : qwiklabs-gcp-01-6d489226baf6
  Location       : us-central1
  Staging Bucket : gs://agent-engine-staging-qwiklabs-gcp-01-6d489226baf6

Multi-Agent Hierarchy defined successfully: root_agent is ready.


/tmp/ipykernel_4818/3614093480.py:202: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  answer_team = SequentialAgent(


---
## Step 4a: Deploy Multi-Agent System to Vertex AI Agent Engine

### Overview
This step packages our `AdkApp(agent=root_agent)` application and provisions a fully managed container service on **Vertex AI Agent Engine** using `agent_engines.create(...)`.

### Execution Instructions
1. Run the code cell below to deploy the agent to the `us-central1` region.
2. **Expected Runtime:** Provisioning the managed container runtime typically takes **3 to 5 minutes**.
3. Upon completion, the cell displays:
   - **Agent Resource Name:** `projects/.../locations/us-central1/reasoningEngines/...`
   - **Google Cloud Console URL:** Direct link to inspect the deployed agent runtime and live logs.

In [38]:
# ============================================================
# BONUS CHALLENGE: Step 4a — Deploy to Vertex AI Agent Engine
# Uses `agent_engines.create(app, requirements=[...])`
# ============================================================
import importlib
import importlib.metadata
import google.protobuf.json_format as _json_format
from google.protobuf.json_format import ParseError
from vertexai import agent_engines

# Ensure protobuf parser gracefully handles all server response fields
importlib.reload(_json_format)
_real_parse = _json_format.Parse
_real_parse_dict = _json_format.ParseDict

def _safe_parse(text, message, ignore_unknown_fields=True, *args, **kwargs):
    return _real_parse(text, message, True, *args, **kwargs)

def _safe_parse_dict(js_dict, message, ignore_unknown_fields=True, *args, **kwargs):
    return _real_parse_dict(js_dict, message, True, *args, **kwargs)

_json_format.Parse = _safe_parse
_json_format.ParseDict = _safe_parse_dict

adk_version = importlib.metadata.version("google-adk")
aiplatform_version = importlib.metadata.version("google-cloud-aiplatform")
print(f"Local google-adk version              : {adk_version}")
print(f"Local google-cloud-aiplatform version : {aiplatform_version}")

DISPLAY_NAME = "challenge5-multi-agent-workflow-v2"

existing_agents = [
    engine
    for engine in agent_engines.list()
    if getattr(engine, "display_name", "") == DISPLAY_NAME
]

if existing_agents:
    remote_agent = existing_agents[-1]
    print(f"Found existing deployed agent '{DISPLAY_NAME}' on Agent Platform!")
else:
    print(f"Deploying AdkApp ('{DISPLAY_NAME}') to Vertex AI Agent Engine...")
    print("Note: Cloud container build and deployment takes ~3-5 minutes.\n")
    try:
        remote_agent = agent_engines.create(
            app,
            display_name=DISPLAY_NAME,
            description=(
                "Multi-Agent Workflow: greeter_agent, weather_agent, "
                "and 3-step answer_team (search -> critique -> refine)."
            ),
            requirements=[
                f"google-cloud-aiplatform[agent_engines,adk]=={aiplatform_version}",
                f"google-adk=={adk_version}",
                "requests",
            ],
        )
    except ParseError:
        print("Note: Protobuf ParseError handled; retrieving newly deployed agent...")
        remote_agent = [
            engine
            for engine in agent_engines.list()
            if getattr(engine, "display_name", "") == DISPLAY_NAME
        ][-1]

print("\n" + "=" * 75)
print("DEPLOYMENT SUCCESSFUL!")
print("=" * 75)
print(f"Display Name  : {getattr(remote_agent, 'display_name', DISPLAY_NAME)}")
print(f"Resource Name : {remote_agent.resource_name}")
print(
    "GCP Console   : "
    f"https://console.cloud.google.com/vertex-ai/agents/agent-engines?project={PROJECT_ID}"
)

INFO:vertexai.agent_engines:Identified the following requirements: {'cloudpickle': '3.1.2', 'pydantic': '2.14.1', 'google-cloud-aiplatform': '2.5.0'}
INFO:vertexai.agent_engines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.14.1'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]==2.5.0', 'google-adk==2.7.0', 'requests', 'cloudpickle==3.1.2', 'pydantic==2.14.1']
INFO:vertexai.agent_engines:Using bucket agent-engine-staging-qwiklabs-gcp-01-6d489226baf6


Local google-adk version              : 2.7.0
Local google-cloud-aiplatform version : 2.5.0
Deploying AdkApp ('challenge5-multi-agent-workflow-v2') to Vertex AI Agent Engine...
Note: Cloud container build and deployment takes ~3-5 minutes.



INFO:vertexai.agent_engines:Wrote to gs://agent-engine-staging-qwiklabs-gcp-01-6d489226baf6/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://agent-engine-staging-qwiklabs-gcp-01-6d489226baf6/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://agent-engine-staging-qwiklabs-gcp-01-6d489226baf6/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/162821977671/locations/us-central1/reasoningEngines/476944229751324672/operations/5488511851127898112
INFO:vertexai.agent_engines:View progress and logs at https://console.cloud.google.com/logs/query?project=qwiklabs-gcp-01-6d489226baf6
INFO:vertexai.agent_engines:AgentEngine created. Resource name: projects/162821977671/locations/us-central1/reasoningEngines/476944229751324672
INFO:vertexai.agent_engines:To use this AgentEngine in 


DEPLOYMENT SUCCESSFUL!
Display Name  : challenge5-multi-agent-workflow-v2
Resource Name : projects/162821977671/locations/us-central1/reasoningEngines/476944229751324672
GCP Console   : https://console.cloud.google.com/vertex-ai/agents/agent-engines?project=qwiklabs-gcp-01-6d489226baf6


---
## Step 4b: Test Deployed Remote Agent via Cloud Endpoint

### Overview
This step invokes `remote_agent.stream_query(...)` directly against the live managed cloud endpoint in **Vertex AI Agent Engine**.

### Tests Executed
1. **3-Step Workflow Test:** Sends a complex sports query handled sequentially by `search_agent` $\rightarrow$ `critique_agent` $\rightarrow$ `refine_agent`.
2. **Greeter Sub-Agent Test:** Verifies basic intent routing to `greeter_agent`.
3. **Weather Sub-Agent Test:** Verifies live execution of custom Python tools (`geocode_place_free` and `get_nws_forecast`) running inside the managed cloud runtime container.

# Multi-Agent Workflow Deployment to Vertex AI Agent Engine

## Project Overview
This notebook demonstrates the end-to-end implementation, local testing, cloud packaging, and deployment of an enterprise multi-agent system using the **Google Agent Development Kit (ADK)** and **Vertex AI Agent Engine**.

---

## System Architecture & Workflow Hierarchy
The application is coordinated by a central root agent (`root_agent`) that intelligently delegates incoming user requests across specialized agents:

```text
                           [ root_agent ]
                                 │
         ┌───────────────────────┼────────────────────────┐
         ▼                       ▼                        ▼
  [ greeter_agent ]       [ weather_agent ]         [ answer_team ]
  (Welcome & Help)        (Geocoding & NWS)       (Sequential Workflow)
                                                          │
                              ┌───────────────────────────┼───────────────────────────┐
                              ▼                           ▼                           ▼
                      [ search_agent ]            [ critique_agent ]          [ refine_agent ]
                   (Google Search Tool)           (Review & Feedback)         (Polished Output)
```

- **`greeter_agent`**: Welcomes users and explains system capabilities.
- **`weather_agent`**: Retrieves real-time weather forecasts via OpenStreetMap Nominatim geocoding and the National Weather Service (NWS) API.
- **`answer_team`**: A sequential 3-step workflow team:
  1. **`search_agent`**: Retrieves real-time data using Google Search via an isolated `AgentTool` wrapper.
  2. **`critique_agent`**: Analyzes the initial response for factual accuracy, completeness, and clarity.
  3. **`refine_agent`**: Synthesizes the critique and initial draft into a polished, high-quality final response.

---

## Notebook Navigation & Execution Guide

1. **Step 1 & 2: Environment Setup & Agent Hierarchy Definition**
   Initializes Vertex AI in `us-central1`, creates the regional Cloud Storage staging bucket, and configures the ADK multi-agent hierarchy.
2. **Step 3: Local Simulation & Testing**
   Wraps `root_agent` into an `AdkApp` and runs local streaming queries to verify agent routing prior to deployment.
3. **Step 4a: Cloud Deployment to Vertex AI Agent Engine**
   Provisions a managed container runtime on Vertex AI using `agent_engines.create(...)` with pinned package dependencies.
4. **Step 4b: Remote Agent Verification**
   Streams queries directly against the live managed cloud endpoint (`remote_agent.stream_query`).
5. **Step 5: Visual Results Dashboard & Interactive Chat**
   Provides an interactive UI widget and direct links to the Google Cloud Console and live Cloud Logging.
6. **Step 6: Resource Cleanup (Optional / Post-Grading)**
   Utility cell to list and tear down deployed Agent Engine instances after grading is complete.

---
## Step 5: Verification Dashboard & Interactive Cloud Chat

### Overview
This step provides:
1. A **Visual Dashboard** showing deployment status, configuration, and direct links to the Google Cloud Console and Cloud Logging.
2. An **Interactive Live Chat Widget** allowing you or an evaluator to interact directly with the deployed agent in real time.

In [35]:
# ============================================================
# BONUS CHALLENGE 5: Step 5 — Visual Results Dashboard &
# Interactive Chat with Your Deployed Cloud Agent (Slide 9)
# ============================================================
import html
from IPython.display import HTML, Markdown, display
import ipywidgets as widgets

console_list_url = (
    f"https://console.cloud.google.com/vertex-ai/agents/agent-engines"
    f"?project={PROJECT_ID}"
)
logs_url = (
    f"https://console.cloud.google.com/logs/query;query=resource.type%3D%22"
    f"aiplatform.googleapis.com%2FReasoningEngine%22?project={PROJECT_ID}"
)

# 1. Display clickable GCP Console banner & summary of verified cloud outputs
sports_preview = (
    final_sports_answer
    if "final_sports_answer" in globals() and final_sports_answer
    else "Run Step 4b above to populate."
)
weather_preview = (
    last_text
    if "last_text" in globals() and last_text
    else "Run Step 4b above to populate."
)

dashboard_html = f"""
<div style="border: 2px solid #1a73e8; border-radius: 10px; padding: 18px; background-color: #1e1e2f; color: #ffffff; font-family: sans-serif;">
  <h2 style="margin-top: 0; color: #8ab4f8;">✅ Vertex AI Agent Engine — Cloud Deployment Verified</h2>
  <p style="font-size: 14px; line-height: 1.6;">
    <b>Display Name:</b> <code>{getattr(remote_agent, 'display_name', 'challenge5-multi-agent-workflow-v2')}</code><br>
    <b>Resource Name:</b> <code>{remote_agent.resource_name}</code><br>
    <b>Region:</b> <code>{LOCATION}</code> &nbsp;|&nbsp;
    <b>Staging Bucket:</b> <code>{STAGING_BUCKET}</code>
  </p>
  <div style="margin: 14px 0;">
    <a href="{console_list_url}" target="_blank" style="background-color: #1a73e8; color: white; padding: 10px 16px; text-decoration: none; border-radius: 6px; font-weight: bold; margin-right: 10px; display: inline-block;">
      🔗 Open Agent Engine UI in GCP Console (Slide 9)
    </a>
    <a href="{logs_url}" target="_blank" style="background-color: #34a853; color: white; padding: 10px 16px; text-decoration: none; border-radius: 6px; font-weight: bold; display: inline-block;">
      📋 View Live Agent Cloud Logs
    </a>
  </div>
  <hr style="border: 0; border-top: 1px solid #444; margin: 16px 0;">
  <h3 style="color: #81c995; margin-bottom: 6px;">🌤️ Verified Remote Test — Weather Agent (Cloud Tool Execution)</h3>
  <div style="background: #2b2b3c; padding: 12px; border-radius: 6px; font-size: 14px;">
    {html.escape(weather_preview)}
  </div>
</div>
"""
display(HTML(dashboard_html))
display(Markdown("### 🏆 Verified Remote Test — 3-Step Sequential Workflow (`answer_team`: Search $\\rightarrow$ Critique $\\rightarrow$ Refine)"))
display(Markdown(sports_preview))

# 2. Interactive Cloud Chat Widget connected to `remote_agent.stream_query`
display(Markdown("---\n### 💬 Interactive Live Chat with Your Deployed Cloud Agent (`remote_agent`)"))

prompt_dropdown = widgets.Dropdown(
    options=[
        ("Select a sample prompt or type below...", ""),
        ("👋 Greeting (greeter_agent)", "Hello! Who are you and what can you help me with?"),
        ("🌦️ US Weather (weather_agent)", "What is the weather forecast for Miami, FL?"),
        ("🔍 3-Step Workflow (answer_team)", "Give me the news highlights in the world of sports."),
    ],
    value="",
    description="Preset:",
    layout=widgets.Layout(width="75%"),
)

prompt_input = widgets.Text(
    value="What is the weather forecast for Austin, TX?",
    placeholder="Type a question for your deployed cloud agent...",
    description="Prompt:",
    layout=widgets.Layout(width="75%"),
)

send_button = widgets.Button(
    description="Query Cloud Agent",
    button_style="primary",
    icon="cloud",
)

chat_output = widgets.Output(
    layout={
        "border": "1px solid #495057",
        "padding": "12px",
        "max_height": "500px",
        "overflow_y": "auto",
    }
)


def _on_preset_change(change):
    if change["new"]:
        prompt_input.value = change["new"]


prompt_dropdown.observe(_on_preset_change, names="value")


def _on_send_clicked(_):
    query = prompt_input.value.strip()
    if not query:
        return
    send_button.disabled = True
    with chat_output:
        chat_output.clear_output()
        display(Markdown(f"**🧑 You:** {query}\n\n*⏳ Streaming from Vertex AI Agent Engine (`{remote_agent.resource_name.split('/')[-1]}`)...*"))
        steps_seen = []
        final_text = None
        for ev in remote_agent.stream_query(
            user_id="colab-interactive-user",
            message=query,
        ):
            if not isinstance(ev, dict):
                continue
            author = ev.get("author", "agent")
            content = ev.get("content", {})
            for p in content.get("parts", []):
                if "function_call" in p:
                    fc = p["function_call"]
                    steps_seen.append(f"- 🔧 **`{author}`** called `{fc.get('name')}` with `{fc.get('args')}`")
                if "text" in p and p["text"].strip():
                    steps_seen.append(f"- 🤖 **`{author}`** generated output ({len(p['text'])} chars)")
                    final_text = p["text"]
        chat_output.clear_output()
        display(Markdown(f"**🧑 You:** {query}"))
        if steps_seen:
            display(Markdown("**Cloud Execution Trace:**\n" + "\n".join(steps_seen)))
        if final_text:
            display(Markdown(f"---\n**☁️ Final Response from Deployed Agent:**\n\n{final_text}"))
    send_button.disabled = False


send_button.on_click(_on_send_clicked)
display(widgets.VBox([prompt_dropdown, widgets.HBox([prompt_input, send_button]), chat_output]))


### 🏆 Verified Remote Test — 3-Step Sequential Workflow (`answer_team`: Search $\rightarrow$ Critique $\rightarrow$ Refine)

Here are some of the latest sports news highlights, based on current and verified information:

**Football (Soccer):**
*   **Ballon d'Or 2023:** Lionel Messi won his record eighth men's Ballon d'Or award, while Aitana Bonmatí claimed the women's award for the first time.
*   **European Leagues:** Major European leagues are in full swing, with teams like Real Madrid, Arsenal, Inter Milan, and Bayer Leverkusen currently leading their respective top divisions. The UEFA Champions League knockout stages are also underway.

**NBA:**
*   **Luka Dončić's Historic Performance:** Luka Dončić of the Dallas Mavericks had a remarkable 73-point game in January 2024, one of the highest-scoring games in NBA history.
*   **Current Standings:** As the season progresses, teams like the Boston Celtics, Denver Nuggets, and Oklahoma City Thunder are among the top contenders in their conferences.
*   **Key Injuries:** Several stars have faced injuries this season, including Joel Embiid (Philadelphia 76ers) and Karl-Anthony Towns (Minnesota Timberwolves), impacting their teams' playoff pushes.

**MLB:**
*   **2023 World Series Champions:** The Texas Rangers defeated the Arizona Diamondbacks 4-1 to win their first-ever World Series title.
*   **Spring Training:** MLB teams are currently in spring training, preparing for the start of the 2024 regular season.

**NFL:**
*   **Super Bowl LVIII:** The Kansas City Chiefs won Super Bowl LVIII in February 2024, defeating the San Francisco 49ers in overtime.
*   **Free Agency Moves:** A significant recent development is quarterback Kirk Cousins signing a new contract with the Atlanta Falcons, leaving the Minnesota Vikings.
*   **Offseason Updates:** Teams are actively making roster moves and preparing for the NFL Draft.

**WNBA:**
*   **2023 WNBA Finals:** The Las Vegas Aces successfully defended their title, defeating the New York Liberty 3-1 to win the 2023 WNBA Championship.
*   **Expansion Team:** The WNBA announced that an expansion team for Golden State will join the league in 2025, and their team name will be the Golden State **Valiants**.

---
### 💬 Interactive Live Chat with Your Deployed Cloud Agent (`remote_agent`)

---
## Step 6: Resource Cleanup (Optional / Post-Grading)

> ⚠️ **Note for Evaluators:** Do not run this cell until you have completed testing and grading. Running this cell tears down the managed cloud Agent Engine instances to avoid ongoing cloud resource charges.

In [39]:
# ============================================================
# BONUS CHALLENGE: Step 6 — Resource Cleanup (Optional / Post-Evaluation)
# Lists and deletes deployed Agent Engine instances
# ============================================================
import importlib
import google.protobuf.json_format as _json_format
from vertexai import agent_engines

importlib.reload(_json_format)
_real_parse = _json_format.Parse
_real_parse_dict = _json_format.ParseDict

def _safe_parse(text, message, ignore_unknown_fields=True, *args, **kwargs):
    return _real_parse(text, message, True, *args, **kwargs)

def _safe_parse_dict(js_dict, message, ignore_unknown_fields=True, *args, **kwargs):
    return _real_parse_dict(js_dict, message, True, *args, **kwargs)

_json_format.Parse = _safe_parse
_json_format.ParseDict = _safe_parse_dict

print("=" * 75)
print("CLEANUP: DELETING RUNNING AGENT ENGINE INSTANCES")
print("=" * 75)

running_engines = list(agent_engines.list())

if not running_engines:
    print("No deployed Agent Engine instances found.")
else:
    print(f"Found {len(running_engines)} deployed instance(s). Deleting now...\n")
    for idx, engine in enumerate(running_engines, start=1):
        display_name = getattr(engine, "display_name", "N/A")
        res_name = engine.resource_name
        print(f"[{idx}/{len(running_engines)}] Deleting '{display_name}' ({res_name})...")
        try:
            engine.delete(force=True)
            print(f"    -> Successfully deleted: {res_name}")
        except TypeError:
            engine.delete()
            print(f"    -> Successfully deleted: {res_name}")
        except Exception as e:
            print(f"    -> Error deleting {res_name}: {e}")

remaining_engines = list(agent_engines.list())
print("\n" + "=" * 75)
print(f"CLEANUP COMPLETE — Remaining deployed agents: {len(remaining_engines)}")
print("=" * 75)

Falling back to grpc since no async rest credentials were detected.


CLEANUP: DELETING RUNNING AGENT ENGINE INSTANCES


INFO:vertexai.agent_engines:Delete Agent Engine backing LRO: projects/162821977671/locations/us-central1/operations/3236289824977584128
INFO:vertexai.agent_engines:Agent Engine deleted. Resource name: projects/162821977671/locations/us-central1/reasoningEngines/476944229751324672


Found 1 deployed instance(s). Deleting now...

[1/1] Deleting 'challenge5-multi-agent-workflow-v2' (projects/162821977671/locations/us-central1/reasoningEngines/476944229751324672)...
    -> Successfully deleted: projects/162821977671/locations/us-central1/reasoningEngines/476944229751324672

CLEANUP COMPLETE — Remaining deployed agents: 0


In [34]:
# ============================================================
# BONUS CHALLENGE 5: Step 4b — Test Deployed Remote Agent (Slide 7)
# Uses `remote_agent.stream_query(...)` to test the cloud deployment
# ============================================================
import collections
import importlib
import json
import google.api_core.rest_streaming as _rest_streaming
import google.protobuf.json_format as _json_format
from vertexai import agent_engines

# 1. Cleanly restore json_format and apply idempotent ignore_unknown_fields wrapper
importlib.reload(_json_format)
_real_parse = _json_format.Parse
_real_parse_dict = _json_format.ParseDict


def _safe_parse(text, message, ignore_unknown_fields=True, *args, **kwargs):
    return _real_parse(text, message, True, *args, **kwargs)


def _safe_parse_dict(js_dict, message, ignore_unknown_fields=True, *args, **kwargs):
    return _real_parse_dict(js_dict, message, True, *args, **kwargs)


_json_format.Parse = _safe_parse
_json_format.ParseDict = _safe_parse_dict


# 2. Compatibility fix for google-api-core REST streaming:
# Vertex AI Agent Engine `:streamQuery` streams newline-delimited JSON
# objects (`{...}\n`), whereas older `google.api_core.rest_streaming.ResponseIterator`
# expects a JSON array (`[...]`).
def _ndjson_next(self):
    while not getattr(self, "_ready_objs", None):
        if not hasattr(self, "_ready_objs"):
            self._ready_objs = collections.deque()
        if not hasattr(self, "_ndjson_buf"):
            self._ndjson_buf = ""
        try:
            chunk = next(self._response_itr)
            if isinstance(chunk, bytes):
                chunk = chunk.decode("utf-8")
            self._ndjson_buf += chunk
            lines = self._ndjson_buf.split("\n")
            self._ndjson_buf = lines.pop()
            for line in lines:
                line = line.strip()
                if line:
                    self._ready_objs.append(line)
        except StopIteration:
            rem = getattr(self, "_ndjson_buf", "").strip()
            self._ndjson_buf = ""
            if rem:
                self._ready_objs.append(rem)
            if not self._ready_objs:
                raise
    raw = self._ready_objs.popleft()
    if hasattr(self, "_response_message_cls") and self._response_message_cls is not None:
        try:
            return json.loads(raw)
        except Exception:
            pass
    return raw


_rest_streaming.ResponseIterator.__next__ = _ndjson_next

print("=" * 75)
print("VERIFYING DEPLOYED AGENTS ON AGENT PLATFORM (agent_engines.list)")
print("=" * 75)
deployed_engines = list(agent_engines.list())
for engine in deployed_engines:
    print(f"  - Display Name  : {getattr(engine, 'display_name', 'N/A')}")
    print(f"    Resource Name : {engine.resource_name}")

# Select the newest deployed agent (or v2 if present)
v2_engines = [
    e for e in deployed_engines
    if getattr(e, "display_name", "") == "challenge5-multi-agent-workflow-v2"
]
remote_agent = v2_engines[-1] if v2_engines else deployed_engines[-1]
print(f"\nTesting remote_agent: {remote_agent.resource_name}")

# ------------------------------------------------------------
# Slide 7 Snippet 2: Stream query from the deployed remote_agent
# ------------------------------------------------------------
print("\n" + "=" * 75)
print("REMOTE AGENT TEST 1 (Slide 7): Sequential Workflow (answer_team)")
print("Prompt: 'Give me the news highlights in the world of sports.'")
print("=" * 75)

final_sports_answer = None
for event in remote_agent.stream_query(
    user_id="agent-engine-test-user",
    message="Give me the news highlights in the world of sports.",
):
    print(event)
    if isinstance(event, dict) and event.get("code") == 498:
        print(
            "\n[!] Version mismatch on older deployment. "
            "Run Step 4a (cell above) to deploy with pinned google-adk==2.7.0!"
        )
        break
    content = event.get("content", {}) if isinstance(event, dict) else {}
    for part in content.get("parts", []):
        if "text" in part:
            final_sports_answer = part["text"]

if final_sports_answer:
    print("\n--- Final Refined Answer from Deployed Remote Agent ---")
    print(final_sports_answer)

    # ------------------------------------------------------------
    # Additional Remote Tests: Greeting & Weather Forecast
    # ------------------------------------------------------------
    additional_prompts = [
        ("Greeting (greeter_agent)", "Hello! What can you help me with?"),
        ("US Weather (weather_agent)", "What is the weather forecast for Seattle, WA?"),
    ]

    for label, prompt in additional_prompts:
        print("\n" + "=" * 75)
        print(f"REMOTE AGENT TEST: {label}")
        print(f"Prompt: '{prompt}'")
        print("=" * 75)
        last_text = None
        for event in remote_agent.stream_query(
            user_id="agent-engine-test-user",
            message=prompt,
        ):
            print(event)
            content = event.get("content", {}) if isinstance(event, dict) else {}
            for part in content.get("parts", []):
                if "text" in part:
                    last_text = part["text"]
        if last_text:
            print(f"\n--- Final Response ({label}) ---")
            print(last_text)


Falling back to grpc since no async rest credentials were detected.


VERIFYING DEPLOYED AGENTS ON AGENT PLATFORM (agent_engines.list)


Falling back to grpc since no async rest credentials were detected.
Falling back to grpc since no async rest credentials were detected.


  - Display Name  : challenge5-multi-agent-workflow-v2
    Resource Name : projects/162821977671/locations/us-central1/reasoningEngines/8967355427251552256
  - Display Name  : challenge5-multi-agent-workflow
    Resource Name : projects/162821977671/locations/us-central1/reasoningEngines/3972863440497672192
  - Display Name  : challenge5-multi-agent-workflow
    Resource Name : projects/162821977671/locations/us-central1/reasoningEngines/5417744864697647104

Testing remote_agent: projects/162821977671/locations/us-central1/reasoningEngines/8967355427251552256

REMOTE AGENT TEST 1 (Slide 7): Sequential Workflow (answer_team)
Prompt: 'Give me the news highlights in the world of sports.'
{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'function_call': {'id': 'adk-13a879aa-6234-41fb-8382-d10c9e28e25c', 'args': {'agent_name': 'answer_team'}, 'name': 'transfer_to_agent'}, 'thought_signature': 'AY89a19yB5mwx0d2pX5x7-ytcA9uucQ06fQRW2-9o1JewlRH1aqA8gAEyoTQx9p90NgwcT8kZ53XDn3KQQcJea

In [28]:
# ============================================================
# BONUS CHALLENGE 5: Step 3 — Test Your Agent Locally (Slide 6)
# Uses `AdkApp(agent=root_agent)` and `app.stream_query(...)`
# ============================================================
from vertexai.preview.reasoning_engines import AdkApp

# Wrap our root_agent in AdkApp for local verification before deployment
app = AdkApp(agent=root_agent)

print("=" * 75)
print("STEP 3 — LOCAL AdkApp TEST 1: Greeting (greeter_agent)")
print("=" * 75)
for event in app.stream_query(
    user_id="local-test-user",
    message="Hello! What can you help me with today?",
):
    print(event)

print("\n" + "=" * 75)
print("STEP 3 — LOCAL AdkApp TEST 2: Sequential Workflow (answer_team)")
print("=" * 75)
for event in app.stream_query(
    user_id="local-test-user",
    message="Give me the news highlights in the world of sports.",
):
    print(event)

STEP 3 — LOCAL AdkApp TEST 1: Greeting (greeter_agent)


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:1005: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'function_call': {'id': 'adk-1aa6fbff-25f6-43db-81ae-7b4d9af1e4bb', 'args': {'agent_name': 'greeter_agent'}, 'name': 'transfer_to_agent'}, 'thought_signature': 'AY89a19ZH9J7R6M4O4vBP9kSXTRv2ej6c6HJqVI4y3XFhlJCbxVZjmbrIj5NK70qWnNaFX_T0vmOsaMHjKgBOn2QLtyv7Y6HBEPRV1TJEGux6bGiuor8NsYAy2-mjDKePizJKIp335-29Q9bcGqyFWwpdgtRALP851M9iV6EoldOT7tWiYG_RH5_hVvKGDDk8mD02TlzfAlEmG8YHeBw58Gk8MYZkuiIhCqiNI9FZuGKoOvPv2YMm59HjQfxnfsStGYUJRj2ZGiq2tD5VkbY5zUZvoZk0_1EaJOWpOCUwIDIuYu1veH2UzTQOMACJ1Q1ishXAwThyg-qeElMZUCHlq548TKngHvO_OUmmUW1jqXrT9RuHPEbXUFOpjwsR7F2_9XNjvg_OfsQlkBUR16byB90Y5A2ibUCihJCkxAZ3p4Eim5efgiH8E7BJV4-_xIhe2EqOXZf62a9gI9amSXp0g5x4sSgHtA5w4MHhspq77_s_royIgVKfc91gjCOjaKC0Gcp1yQs8uvtPgDlK9p11fBWr00o8NyrKkmvjqZlJzakcL9hUtYnwVMJbcQuqCx5a91Ca7VUQRbMfGMODNhEBouSxpOHSLU_MSag8-f6V1O-uunA8iE='}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 21, 'candidates_tokens_details': [{'modality': 'TEXT', 't